# ReGLU — ReLU-Gated Linear Unit

源码导航：[core/ffn/reglu.py](../../../core/ffn/reglu.py) 中的 `ReGLUMLP`。

Shazeer (2020) 在 *GLU Variants Improve Transformer* 中提出的 ReGLU 变体，将门控激活函数替换为 ReLU。与 SwiGLU / GEGLU 的平滑门控不同，ReGLU 的门控信号是"硬开关"：负值区完全截断为 0，正值区线性通过。

### 1. 理论推导

ReGLU 公式：

$$\text{ReGLU}(x) = \text{ReLU}(x W_{\text{gate}}) \odot (x W_{\text{up}}) \; W_{\text{down}}$$

**ReLU 门控的特性**：
- 门控输出严格非负，不存在负值渗漏；
- 计算最简单（仅需 max 操作），推理速度略快；
- 训练时容易出现"死门控"：若 gate_proj 的权重学习不当，大量输出恒为 0，对应 up 通道信息永久丢失；
- 因此 ReGLU 在大型 Transformer 中的采用率低于 SwiGLU / GEGLU。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.ffn.reglu import ReGLUMLP

### 2. 形状与 dtype 检查

In [ ]:
torch.manual_seed(0)
mlp = ReGLUMLP(n_embd=128, d_ffn=256, dropout=0.0, bias=False)

x = torch.randn(2, 8, 128)
y = mlp(x)

print(f"输入形状: {tuple(x.shape)}")
print(f"输出形状: {tuple(y.shape)}")
assert x.shape == y.shape, "ReGLU 必须保持输入输出维度一致！"
"参数量:", sum(p.numel() for p in mlp.parameters()))

### 3. 门控信号硬截断验证

验证 ReGLU 的门控输出确实不存在负值，并与 SwiGLU 做对比。

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)
x = torch.randn(2, 16, 128)

reglu = ReGLUMLP(n_embd=128, d_ffn=256, bias=False)

with torch.no_grad():
    gate = torch.nn.functional.relu(reglu.gate_proj(x))

plt.figure(figsize=(8, 4))
plt.hist(gate.flatten().numpy(), bins=50, color="orange", alpha=0.7)
plt.title("ReGLU gate output distribution")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.axvline(0, color="red", linestyle="--")
plt.tight_layout()
plt.show()

print(f"最小值: {gate.min().item():.4f}（应 >= 0）")
print(f"零值比例: {(gate == 0).float().mean().item():.4f}")
assert gate.min().item() >= -1e-6, "ReLU 门控不应有负值"

### 4. 源码精讲

```python
class ReGLUMLP(nn.Module):
    def __init__(self, n_embd, d_ffn, dropout=0.0, bias=False):
        super().__init__()
        self.gate_proj = nn.Linear(n_embd, d_ffn, bias=bias)
        self.up_proj = nn.Linear(n_embd, d_ffn, bias=bias)
        self.down_proj = nn.Linear(d_ffn, n_embd, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gate = F.relu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.dropout(self.down_proj(gate * up))
```

关键差异：第 9 行使用 `F.relu`，使门控成为硬开关。这种设计在部分轻量级模型或特定硬件上有速度优势，但大型语言模型中较少采用。

---

## 延伸阅读与参考资料

### 核心论文
- **GLU Variants Improve Transformer**: Shazeer, 2020. [arXiv:2002.05202](https://arxiv.org/abs/2002.05202)

### 工程讨论
- ReGLU 的硬截断特性使其推理时计算最简单，但训练稳定性不如 SwiGLU / GEGLU。实践中，SwiGLU 凭借 SiLU 的平滑渗漏区成为 LLaMA / Mistral 等模型的默认选择。